In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()
CACHE_DIR = os.getenv("CACHE_DIR")
PARENT_DIR = os.getenv("PARENT_DIR")

In [ ]:
from transformers import AutoTokenizer

In [ ]:
os.environ["VLLM_CPU_KVCACHE_SPACE"] = "4"
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "1"
# os.environ["VLLM_LOGGING_LEVEL"] = "DEBUG"

In [ ]:

prompts = [
    "Write a haiku about city rain.",
    "Describe your ideal breakfast in three vivid sentences. breakfast in three vivid sentences.",
    "Summarize this week",
    "Explain blockchain to a teenager using sports analogies. blockchain to a teenager using sports analogies. blockchain to a teenager using sports analogies.",
    "Draft a polite message asking to reschedule tomorrow's meeting. Draft a polite message asking to reschedule tomorrow's meeting. Draft a polite message asking to reschedule tomorrow's meeting.",
    "Give five quick tips to improve sleep quality tonight. sleep quality tonight.",
    "Invent a product name for smart reusable water bottles.",
    "Plan a two-hour study session for calculus practice. Plan a two-hour study session for calculus practice.",
    "Write a short dialogue between a robot and gardener. robot and gardener.",
    "Create a travel checklist for ",
]

estimated_token_counts = [len(p.split()) for p in prompts]

prompts_labeled = [
    {"estimated_tokens": c, "prompt": p}
    for c, p in zip(estimated_token_counts, prompts)
]

prompts_labeled

In [ ]:
import random
import json
# Create random prompts of varying lengths for testing (random with script)
init_prompt = [
    "Write a haiku about city rain.",
    "Describe your ideal breakfast in three vivid sentences. breakfast in three vivid sentences.",
    "Summarize this week",
    "Explain blockchain to a teenager using sports analogies. blockchain to a teenager using sports analogies. blockchain to a teenager using sports analogies.",
    "Draft a polite message asking to reschedule tomorrow's meeting. Draft a polite message asking to reschedule tomorrow's meeting. Draft a polite message asking to reschedule tomorrow's meeting.",
    "Give five quick tips to improve sleep quality tonight. sleep quality tonight.",
    "Invent a product name for smart reusable water bottles.",
    "Plan a two-hour study session for calculus practice. Plan a two-hour study session for calculus practice.",
    "Write a short dialogue between a robot and gardener. robot and gardener.",
    "Create a travel checklist for ",
]

prompts = []
n_prompts = 1000
n_test_cases = [10, 20, 50, 100, 200, 500, 1000]
for i in range(n_prompts):
    # Randomly select a base prompt and append a random number of words to increase length
    base_prompt = random.choice(init_prompt)
    num_words = random.randint(5, 100)  # Randomly add between 5 and 100 words
    words = [f"word_{j}" for j in range(num_words)]
    prompts.append(f"{base_prompt} {' '.join(words)}")

    # Store to a json file if i in n_test_cases for later analysis
    if i + 1 in n_test_cases:
        with open(f"{PARENT_DIR}/test_cases/prompts_{i + 1}.json", "w", encoding="utf-8") as f:
            json.dump(prompts, f, indent=4, ensure_ascii=False)

In [ ]:
# main.py
from vllm.v1.core.sched.scheduler import Scheduler
from vllm.v1.core.sched.request_queue import SchedulingPolicy, create_request_queue

_orig_init = Scheduler.__init__
_orig_add_request = Scheduler.add_request

def _lpf_prompt_len(req):
    # Prefer explicit count already maintained by Request.
    # Fallback to prompt token list length if needed.
    n = getattr(req, "num_prompt_tokens", None)
    if n is None:
        pt = getattr(req, "prompt_token_ids", None) or []
        n = len(pt)
    return int(n)

def patched_init(self, *args, **kwargs):
    _orig_init(self, *args, **kwargs)

    # Force priority mode so queue ordering/preemption uses Request.priority.
    self.policy = SchedulingPolicy.PRIORITY
    self.waiting = create_request_queue(self.policy)
    self.skipped_waiting = create_request_queue(self.policy)

def patched_add_request(self, request):
    # LPF: larger prompt should run earlier; smaller priority value wins.
    # Use negative length because PRIORITY pops smallest first.
    prompt_len = _lpf_prompt_len(request)
    request.priority = -prompt_len

    return _orig_add_request(self, request)

Scheduler.__init__ = patched_init
Scheduler.add_request = patched_add_request

In [ ]:
from vllm import LLM
llm = LLM(
    model="Qwen/Qwen2-0.5B",
    download_dir=CACHE_DIR, # Set your runtime cache directory here
    max_model_len=1024,   # reduce KV cache demand
    enforce_eager=True,
    max_num_seqs=1 # 
)

In [ ]:
from vllm import SamplingParams
import random
import time


def print_table(title, columns, data_rows):
    print(f"\n===== {title} =====")
    widths = []
    for i, col in enumerate(columns):
        max_data_len = max((len(str(row[i])) for row in data_rows), default=0)
        widths.append(max(len(col), max_data_len))

    header = " | ".join(col.ljust(widths[i]) for i, col in enumerate(columns))
    sep = "-+-".join("-" * widths[i] for i in range(len(columns)))
    print(header)
    print(sep)
    for row in data_rows:
        print(" | ".join(str(row[i]).ljust(widths[i]) for i in range(len(columns))))


def fmt_float(x):
    return f"{x:.4f}" if x is not None else "n/a"


# Build a shuffled batch so scheduling order is not the same as prompt length order.
indexed_prompts = list(enumerate(prompts))
random.seed(42)
random.shuffle(indexed_prompts)

orig_indices = [i for i, _ in indexed_prompts]
batched_prompts = [p for _, p in indexed_prompts]
estimated_by_orig_idx = {i: len(prompts[i].split()) for i in range(len(prompts))}

input_rows = [
    (pos + 1, orig_idx, estimated_by_orig_idx[orig_idx])
    for pos, orig_idx in enumerate(orig_indices)
]
print_table(
    "Input Batch Order",
    ("Order", "Original ID", "Prompt Length"),
    input_rows,
 )

sampling_params = SamplingParams(temperature=0.0, max_tokens=32)
t0 = time.perf_counter()
outputs = llm.generate(batched_prompts, sampling_params, use_tqdm=False)
t1 = time.perf_counter()

rows = []
for pos, out in enumerate(outputs):
    orig_idx = orig_indices[pos]
    est_prompt_tokens = estimated_by_orig_idx[orig_idx]
    text = out.outputs[0].text if out.outputs else ""
    preview = text.strip().replace("\n", " ")[:60]

    metrics = getattr(out, "metrics", None)
    first_token_time = getattr(metrics, "first_token_time", None) if metrics else None
    finished_time = getattr(metrics, "finished_time", None) if metrics else None
    arrival_time = getattr(metrics, "arrival_time", None) if metrics else None

    ttft = (first_token_time - arrival_time) if (first_token_time and arrival_time) else None
    total_latency = (finished_time - arrival_time) if (finished_time and arrival_time) else None

    rows.append(
        {
            "order": pos + 1,
            "orig_idx": orig_idx,
            "est_prompt_tokens": est_prompt_tokens,
            "ttft": ttft,
            "total_latency": total_latency,
            "finished_time": finished_time,
            "preview": preview,
        }
    )

observed_rows = [
    (
        r["order"],
        r["orig_idx"],
        r["est_prompt_tokens"],
        fmt_float(r["ttft"]),
        fmt_float(r["total_latency"]),
        r["preview"],
    )
    for r in rows
]
print_table(
    "Observed Outputs (Input Order)",
    ("Order", "Original ID", "Prompt Length", "TTFT (s)", "Total (s)", "Preview"),
    observed_rows,
 )

if any(r["finished_time"] is not None for r in rows):
    completion_order = sorted(
        rows,
        key=lambda r: (r["finished_time"] is None, r["finished_time"] if r["finished_time"] is not None else float("inf")),
    )
    completion_rows = [
        (rank, r["orig_idx"], r["est_prompt_tokens"], r["finished_time"])
        for rank, r in enumerate(completion_order, 1)
    ]
    print_table(
        "Approx Completion Order (by finished_time)",
        ("Order", "Original ID", "Prompt Length", "finished_time"),
        completion_rows,
    )
else:
    print("\nNo per-request finish timestamps were available in output metrics.")

print(f"\nBatch wall time: {t1 - t0:.4f}s")
print("\nTip: Under LPF, longer prompts should trend toward earlier completion/service order.")